In [74]:
from PIL import Image
import os
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score

img = Image.open("training_data/train_img_01.jpg") #open an image
tiles_folder = "train_tiles"

tile = img.crop((0, 0, 128, 128)) #get a tile from image
tile_path = os.path.join(tiles_folder, "img1_tile0.jpg") 
tile.save(tile_path) #save to path

tile = img.crop((128, 0, 256, 128)) #once more
tile_path = os.path.join(tiles_folder, "img1_tile1.jpg")
tile.save(tile_path)

In [75]:
tile1 = [] #name of first tile
tile2 = [] #name of second tile
answer = [] #labels
tile1.append("img1_tile0.jpg") 
tile2.append("img1_tile1.jpg")
answer.append(1) #tile1 to left of tile2

tile1.append("img1_tile1.jpg")
tile2.append("img1_tile0.jpg")
answer.append(3) #tile1 to left of tile2


# Validation

In [76]:
# binary_labels = np.array(validation_df['answer'] > 0, dtype=int)
# false_positive_rate, true_positive_rate, thresholds = roc_curve(binary_labels, preds)
# roc_auc = auc(false_positive_rate, true_positive_rate)

# plt.figure()
# plt.plot(false_positive_rate, true_positive_rate, label=f"ROC curve (AUC = {roc_auc:.2f})")
# plt.plot([0, 1], [0, 1], linestyle="--")  # random baseline

# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.title("ROC Curve")
# plt.legend()
# plt.show()

In [77]:
# binary_preds = preds < 0.77
# f1_score(binary_labels, binary_preds, average='macro')

# Train data generation

In [78]:
import random

from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

IMG_SIZE = 512
TILE_SIZE = 128
GRID = 4

train_images_folder = "training_data"
tiles_folder = "train_tiles"
os.makedirs(tiles_folder, exist_ok=True)

In [79]:
image_to_tiles = {}  # image_id -> list of tile filenames

def extract_tiles():
    for img_name in tqdm(os.listdir(train_images_folder)):
        if not img_name.endswith(".jpg"):
            continue

        img_path = os.path.join(train_images_folder, img_name)
        img = Image.open(img_path).convert("RGB")

        image_id = img_name.split(".")[0]
        tiles = []

        for i in range(GRID):
            for j in range(GRID):
                left = j * TILE_SIZE
                top = i * TILE_SIZE
                right = left + TILE_SIZE
                bottom = top + TILE_SIZE

                tile = img.crop((left, top, right, bottom))

                tile_name = f"{image_id}_tile_{i}_{j}.jpg"
                tile_path = os.path.join(tiles_folder, tile_name)

                tile.save(tile_path)
                tiles.append((tile_name, i, j))  # store position

        image_to_tiles[image_id] = tiles


extract_tiles()

100%|██████████| 60/60 [00:01<00:00, 30.31it/s]


In [80]:
tile1_list = []
tile2_list = []
labels = []

image_ids = list(image_to_tiles.keys())

def get_direction(a, b):
    # a, b are (i, j)
    ai, aj = a
    bi, bj = b

    if ai == bi and aj + 1 == bj:
        return 1  # a left of b
    if ai == bi and aj - 1 == bj:
        return 3  # a right of b
    if aj == bj and ai + 1 == bi:
        return 2  # a above b
    if aj == bj and ai - 1 == bi:
        return 4  # a below b
    return None

In [81]:
for img_id, tiles in tqdm(image_to_tiles.items()):

    # split tiles into positions
    pos_map = {t[0]: (t[1], t[2]) for t in tiles}
    tile_names = [t[0] for t in tiles]

    # SAME IMAGE PAIRS
    for i in range(len(tile_names)):
        for j in range(len(tile_names)):

            if i == j:
                continue

            t1 = tile_names[i]
            t2 = tile_names[j]

            p1 = pos_map[t1]
            p2 = pos_map[t2]

            direction = get_direction(p1, p2)

            tile1_list.append(t1)
            tile2_list.append(t2)

            if direction is not None:
                labels.append(direction)
            else:
                labels.append(5)

100%|██████████| 60/60 [00:00<00:00, 6150.61it/s]


In [82]:
for _ in range(len(tile1_list) // 5):  # control imbalance
    img_a, img_b = random.sample(image_ids, 2)

    tile_a = random.choice(image_to_tiles[img_a])[0]
    tile_b = random.choice(image_to_tiles[img_b])[0]

    tile1_list.append(tile_a)
    tile2_list.append(tile_b)
    labels.append(0)

In [83]:
train_transforms = T.Compose([
    T.Resize((128, 128)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])
val_transforms = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

class TilePairDataset(Dataset):
    def __init__(self, tile1, tile2, labels, folder, transform=None):
        self.tile1 = tile1
        self.tile2 = tile2
        self.labels = labels
        self.folder = folder
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        img1_path = os.path.join(self.folder, self.tile1[idx])
        img2_path = os.path.join(self.folder, self.tile2[idx])

        img1 = Image.open(img1_path).convert("RGB")
        img2 = Image.open(img2_path).convert("RGB")

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return img1, img2, label

In [84]:
dataset = TilePairDataset(
    tile1_list,
    tile2_list,
    labels,
    tiles_folder,
    transform=train_transforms
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
)

for img1, img2, y in loader:
    print(img1.shape)  # [B, 3, 128, 128]
    print(img2.shape)  # [B, 3, 128, 128]
    print(y.shape)     # [B]
    break

torch.Size([32, 3, 128, 128])
torch.Size([32, 3, 128, 128])
torch.Size([32])


In [85]:
VALIDATION_PATH = 'validation_data/validation/validation_tiles/'

validation_df = pd.read_csv(
    'validation_data/validation/validation_data.csv'
)
validation_df['tile1'] = validation_df['tile1'].astype(str) + '.jpg'
validation_df['tile2'] = validation_df['tile2'].astype(str) + '.jpg'

val_dataset = TilePairDataset(
    tile1=validation_df['tile1'].astype(str).values,
    tile2=validation_df['tile2'].astype(str).values,
    labels=validation_df['answer'].values,
    folder=VALIDATION_PATH,
    transform=val_transforms
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    drop_last=True
)

# Clever way

In [86]:
from torchvision import models
from torch import nn
device = 'cuda'

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) 
model.fc = nn.Identity()
for param in model.parameters():
    param.requires_grad = False 
model = model.to(device)

classification_model = nn.Sequential(
    nn.Linear(4*512, 512),
    nn.ReLU(True),
    nn.Linear(512, 128),
    nn.ReLU(True),
    nn.Linear(128, 6)
)
classification_model = classification_model.to(device)

In [87]:
criterion = nn.CrossEntropyLoss()
optim = torch.optim.AdamW(classification_model.parameters(), lr=1e-4)
epochs = 10
val_labels = torch.tensor(validation_df['answer'].tolist())

for i in tqdm(range(epochs)):
    model.eval()
    classification_model.train()
    binary_acc, full_acc = 0.0, 0.0

    for tiles1, tiles2, labels in loader:
        optim.zero_grad()
        tiles1, tiles2, labels = tiles1.to(device), tiles2.to(device), labels.to(device)
        with torch.no_grad():
            emb1s = nn.functional.normalize(model(tiles1), dim=1)
            emb2s = nn.functional.normalize(model(tiles2), dim=1)
        features = torch.cat([emb1s, emb2s, emb1s-emb2s, emb1s * emb2s], dim=1) 
        logits = classification_model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optim.step()

        preds = torch.argmax(logits, dim=1)
        binary_acc += torch.sum((preds > 0) == (labels > 0)).item()
        full_acc += torch.sum(preds == labels).item()
    
    binary_acc/=len(dataset)
    full_acc/=len(dataset)

    val_binary_acc, val_full_acc = 0.0, 0.0
    classification_model.eval()
    with torch.no_grad():
        for tils1, tiles2, labels in val_loader:
            optim.zero_grad()
            tiles1, tiles2, labels = tiles1.to(device), tiles2.to(device), labels.to(device)
            emb1s = nn.functional.normalize(model(tiles1), dim=1)
            emb2s = nn.functional.normalize(model(tiles2), dim=1)
            features = torch.cat([emb1s, emb2s, emb1s-emb2s, emb1s * emb2s], dim=1)

            logits = classification_model(features)

            preds = torch.argmax(logits, dim=1)
            val_binary_acc += torch.sum((preds > 0) == (labels > 0)).item()
            val_full_acc += torch.sum(preds == labels).item()

    val_binary_acc/=len(val_dataset)
    val_full_acc/=len(val_dataset)

    print(f"Epoch {i}/{epochs} | Train binary acc: {binary_acc:.2f} | Train full acc: {full_acc:.2f} | Val binary acc: {val_binary_acc} | Val full acc: {val_full_acc}")

 10%|█         | 1/10 [01:13<11:03, 73.69s/it]

Epoch 0/10 | Train binary acc: 0.83 | Train full acc: 0.66 | Val binary acc: 0.5736842105263158 | Val full acc: 0.28299595141700407


 10%|█         | 1/10 [01:47<16:03, 107.04s/it]


KeyboardInterrupt: 